# 00 Generate source data

Simulates the exports that land in OneLake from the systems of **Nordklang**, a fictional
premium audio brand with its own stores, partner stores and two webshops.

| Source | Format | What it looks like |
|---|---|---|
| Store tills (POS) | CSV, `;` separated, decimal comma or dot | one line per item on a receipt, returns as negative quantity |
| Webshop | JSON Lines | one order per line with nested order lines; an order can be exported again when its status changes |
| Master data | CSV snapshots | products, stores, exchange rates (bank days only), monthly targets per city |

Everything is synthetic and reproducible: each day is generated from its own seed, so a
backfill and a day by day run produce exactly the same data.

Some problems are planted on purpose (see `PLANTED`), so the silver layer and the data
quality gate have something real to catch.

In [ ]:
mode = "backfill"            # "backfill": history up to end_date, "next_day": add one more day
end_date = "2026-10-17"      # last day of history in backfill mode
landing_root = "/lakehouse/default/Files/landing"
seed = 42

In [ ]:
import csv
import io
import json
import math
import os
import random
import datetime as dt

START = dt.date(2024, 6, 1)  # first day of fiscal year 2024/25 (June to May)

# DKK per 1 unit of currency, starting point of the random walk
FX_BASE = {"NOK": 0.635, "SEK": 0.655, "EUR": 7.460, "GBP": 8.620, "USD": 6.880}

# code, name, city, country, region, channel, currency, opened, closed, size, decimal
STORES = [
    ("CPH01", "Nordklang Stroget", "Copenhagen", "Denmark", "Nordics", "Own store", "DKK", "2015-03-01", None, 1.6, ","),
    ("CPH02", "Nordklang Lyngby", "Copenhagen", "Denmark", "Nordics", "Partner store", "DKK", "2018-09-01", None, 0.8, ","),
    ("AAR01", "Nordklang Aarhus", "Aarhus", "Denmark", "Nordics", "Own store", "DKK", "2017-05-01", None, 1.0, ","),
    ("OSL01", "Nordklang Oslo", "Oslo", "Norway", "Nordics", "Partner store", "NOK", "2019-02-01", None, 0.9, ","),
    ("STO01", "Nordklang Stockholm", "Stockholm", "Sweden", "Nordics", "Own store", "SEK", "2016-10-01", None, 1.2, ","),
    ("HAM01", "Nordklang Hamburg", "Hamburg", "Germany", "DACH", "Partner store", "EUR", "2018-04-01", "2025-08-31", 0.7, "."),
    ("MUC01", "Nordklang Munich", "Munich", "Germany", "DACH", "Own store", "EUR", "2019-11-01", None, 1.3, ","),
    ("LON01", "Nordklang Covent Garden", "London", "United Kingdom", "UK", "Own store", "GBP", "2014-06-01", None, 1.8, "."),
    ("LON02", "Nordklang Westfield", "London", "United Kingdom", "UK", "Partner store", "GBP", "2020-01-15", "2025-12-31", 0.8, "."),
    ("PAR01", "Nordklang Le Marais", "Paris", "France", "Southern Europe", "Own store", "EUR", "2017-03-01", None, 1.4, ","),
    ("MIL01", "Nordklang Brera", "Milan", "Italy", "Southern Europe", "Partner store", "EUR", "2021-05-01", None, 0.9, "."),
    ("NYC01", "Nordklang SoHo", "New York", "United States", "Americas", "Own store", "USD", "2019-09-01", None, 1.7, "."),
    ("NYC02", "Nordklang Hudson Yards", "New York", "United States", "Americas", "Own store", "USD", "2025-09-15", None, 1.2, "."),
    ("WEB-DK", "Nordklang Webshop DK", "Online", "Denmark", "Online", "Online", "DKK", "2014-01-01", None, None, None),
    ("WEB-EU", "Nordklang Webshop EU", "Online", "Europe", "Online", "Online", "EUR", "2014-01-01", None, None, None),
]

# yearly growth of demand per city: some cities grow fast, some shrink
CITY_GROWTH = {
    "Copenhagen": 0.04, "Aarhus": 0.01, "Oslo": -0.03, "Stockholm": 0.02, "Hamburg": -0.10,
    "Munich": 0.00, "London": 0.10, "Paris": 0.08, "Milan": 0.05, "New York": 0.15, "Online": 0.12,
}
# growth the business planned when it set the targets
TARGET_GROWTH = 0.06

# sku, name, category, subcategory, list price DKK, cost ratio, launch, discontinued, weight
PRODUCTS = [
    ("NK-SPK-101", "Klang One Portable", "Speakers", "Portable", 2999, 0.42, None, None, 9),
    ("NK-SPK-102", "Klang Go", "Speakers", "Portable", 1999, 0.44, None, None, 10),
    ("NK-SPK-201", "Klang Pillar", "Speakers", "Home", 8499, 0.40, None, None, 5),
    ("NK-SPK-202", "Klang Stage Soundbar", "Speakers", "Home", 12999, 0.41, None, None, 4),
    ("NK-SPK-301", "Klang Monolith", "Speakers", "Floor", 44999, 0.38, None, None, 0.6),
    ("NK-SPK-302", "Klang Monolith Mini", "Speakers", "Floor", 24999, 0.39, "2025-04-01", None, 0.8),
    ("NK-HP-101", "Hush 9", "Headphones", "Over-ear", 4499, 0.43, None, "2026-02-28", 8),
    ("NK-HP-102", "Hush 11", "Headphones", "Over-ear", 5499, 0.42, "2025-10-01", None, 9),
    ("NK-HP-201", "Hush Sport", "Headphones", "On-ear", 2799, 0.45, None, None, 6),
    ("NK-HP-301", "Pebble Buds", "Headphones", "In-ear", 2199, 0.46, None, None, 12),
    ("NK-HP-302", "Pebble Buds Pro", "Headphones", "In-ear", 3299, 0.44, "2026-03-01", None, 11),
    ("NK-TV-155", "Vista 55", "Televisions", "OLED", 39999, 0.47, None, None, 0.7),
    ("NK-TV-165", "Vista 65", "Televisions", "OLED", 64999, 0.46, None, None, 0.5),
    ("NK-TV-177", "Vista 77", "Televisions", "OLED", 109999, 0.45, None, None, 0.2),
    ("NK-ACC-101", "Speaker Cable 3 m", "Accessories", "Cables", 449, 0.30, None, None, 7),
    ("NK-ACC-102", "Wall Bracket", "Accessories", "Mounts", 899, 0.33, None, None, 4),
    ("NK-ACC-103", "Floor Stand", "Accessories", "Mounts", 1999, 0.35, None, None, 2),
    ("NK-ACC-104", "Leather Ear Cushions", "Accessories", "Spare parts", 599, 0.28, None, None, 6),
    ("NK-ACC-105", "Charging Case", "Accessories", "Spare parts", 749, 0.31, None, None, 5),
    ("NK-ACC-106", "Travel Pouch", "Accessories", "Spare parts", 349, 0.25, None, None, 6),
]

# what was planted on purpose and how many rows each problem has
PLANTED = {
    "pos_resend": "the POS file for March 2025 is delivered twice (pos_2025-03_resend.csv)",
    "store_code_format": 0,   # Aarhus lines in February 2025 written as ' aar01'
    "unknown_store": 8,       # training till TRN01, not in the store master
    "unknown_sku": 6,         # test SKU NK-HP-999 sold in January 2026
    "zero_quantity": 5,
    "missing_price": 4,
    "before_store_opened": 3, # staff test sales at NYC02 three days before it opened
    "fx_gap": "no GBP rate from 24 to 28 December 2025",
    "product_master_format": "NK-ACC-104 delivered as 'nk-acc-104 ' with category 'accessories'",
    "web_status_updates": 0,  # orders exported as pending and again the next day with the final status
}


def d(s):
    return dt.date.fromisoformat(s) if s else None


def rng_for(day, salt):
    return random.Random(f"{seed}-{salt}-{day.isoformat()}")


def poisson(r, lam):
    # Knuth; lam is small (< 30) so this is fine
    limit, k, p = math.exp(-lam), 0, 1.0
    while True:
        p *= r.random()
        if p <= limit:
            return k
        k += 1


def season(day):
    factor = {1: 0.70, 2: 0.75, 3: 0.85, 4: 0.90, 5: 0.95, 6: 0.90, 7: 0.80, 8: 0.90,
              9: 1.00, 10: 1.05, 11: 1.45, 12: 1.90}[day.month]
    return factor * (1.6 if is_black_week(day) else 1.0)


def is_black_week(day):
    # Monday to Sunday around Black Friday (fourth Friday of November)
    if day.month != 11:
        return False
    first = dt.date(day.year, 11, 1)
    fridays = [first + dt.timedelta(days=i) for i in range(30) if (first + dt.timedelta(days=i)).weekday() == 4]
    bf = fridays[3]
    return bf - dt.timedelta(days=4) <= day <= bf + dt.timedelta(days=2)


def weekday_factor(day, online=False):
    if online:
        return [1.10, 1.05, 1.0, 1.0, 0.95, 0.90, 1.05][day.weekday()]
    return [0.75, 0.80, 0.85, 0.95, 1.15, 1.55, 0.70][day.weekday()]


def growth(city, day):
    years = (day - START).days / 365.25
    return (1 + CITY_GROWTH[city]) ** years


def is_open(store, day):
    opened, closed = d(store[7]), d(store[8])
    return opened <= day and (closed is None or day <= closed)


def available(product, day):
    launch, disc = d(product[6]), d(product[7])
    return (launch is None or launch <= day) and (disc is None or day <= disc)


def local_price(list_dkk, currency):
    if currency == "DKK":
        return float(list_dkk)
    x = list_dkk / FX_BASE[currency]
    return float(round(x / 10) * 10 - 1) if x >= 100 else float(round(x))


def discount_for(r, product, day):
    if is_black_week(day) and product[2] in ("Speakers", "Headphones"):
        return 15
    if day.month == 1 and day.day <= 14 and r.random() < 0.3:
        return 10
    return 10 if r.random() < 0.02 else 0  # occasional staff or price match discount


def fmt_amount(x, decimal):
    s = f"{x:.2f}"
    return s.replace(".", ",") if decimal == "," else s


def pick_product(r, day):
    items = [p for p in PRODUCTS if available(p, day)]
    return r.choices(items, weights=[p[8] for p in items], k=1)[0]

## One day of data

In [ ]:
POS_HEADER = ["receipt_no", "line_no", "store_code", "sold_at", "sku", "quantity", "unit_price", "discount_pct", "currency"]


def pos_lines_for_day(day):
    rows = []
    for st in STORES:
        if st[5] == "Online" or not is_open(st, day):
            continue
        r = rng_for(day, st[0])
        lam = 4.5 * st[9] * season(day) * weekday_factor(day) * growth(st[2], day)
        for i in range(poisson(r, lam)):
            receipt = f"{st[0]}-{day:%y%m%d}-{i + 1:04d}"
            ts = dt.datetime.combine(day, dt.time(10)) + dt.timedelta(minutes=r.randint(0, 540))
            n_lines = 1 + (r.random() < 0.35) + (r.random() < 0.10)
            is_return = r.random() < 0.03
            for ln in range(1, n_lines + 1):
                p = pick_product(r, day)
                qty = r.randint(1, 3) if p[2] == "Accessories" else 1
                if is_return:
                    qty = -qty
                disc = 0 if is_return else discount_for(r, p, day)
                rows.append([receipt, str(ln), st[0], ts.strftime("%d-%m-%Y %H:%M"), p[0], str(qty),
                             fmt_amount(local_price(p[4], st[6]), st[10]), str(disc), st[6]])
                if is_return:
                    break  # a return receipt has one line
    return rows


def web_orders_created(day):
    """Orders created on this day, each with its final status and whether it was first exported as pending."""
    r = rng_for(day, "web")
    lam = 22 * season(day) * weekday_factor(day, online=True) * growth("Online", day)
    orders = []
    for i in range(poisson(r, lam)):
        dk = r.random() < 0.45
        currency = "DKK" if dk else "EUR"
        market = "DK" if dk else r.choice(["DE", "FR", "NL", "SE", "IT"])
        created = dt.datetime.combine(day, dt.time(0)) + dt.timedelta(minutes=r.randint(0, 1430))
        lines = []
        for ln in range(1, 1 + 1 + (r.random() < 0.25) + (r.random() < 0.05)):
            p = pick_product(r, day)
            lines.append({"line_no": ln, "sku": p[0], "qty": r.randint(1, 2) if p[2] == "Accessories" else 1,
                          "unit_price": round(local_price(p[4], currency) * (1 - discount_for(r, p, day) / 100), 2)})
        final = "cancelled" if r.random() < 0.07 else "completed"
        pending_first = r.random() < 0.20
        orders.append({"order_id": f"W{day:%y%m%d}{i + 1:04d}", "created_at": created.strftime("%Y-%m-%dT%H:%M:%SZ"),
                       "market": market, "currency": currency, "final_status": final,
                       "pending_first": pending_first, "lines": lines})
    return orders


def web_export_for_day(day):
    """What the webshop exports on this day: new orders, plus the final version of yesterday's pending orders."""
    out = []
    exported = dt.datetime.combine(day, dt.time(23, 55)).strftime("%Y-%m-%dT%H:%M:%SZ")
    for o in web_orders_created(day):
        status = "pending" if o["pending_first"] else o["final_status"]
        out.append(_order_json(o, status, exported))
    prev = day - dt.timedelta(days=1)
    if prev >= START:
        update_ts = dt.datetime.combine(day, dt.time(6, 0)).strftime("%Y-%m-%dT%H:%M:%SZ")
        for o in web_orders_created(prev):
            if o["pending_first"]:
                out.append(_order_json(o, o["final_status"], update_ts))
    return out


def _order_json(o, status, exported_at):
    return {"order_id": o["order_id"], "exported_at": exported_at, "created_at": o["created_at"],
            "market": o["market"], "currency": o["currency"], "status": status, "lines": o["lines"]}

## Planted problems

In [ ]:
def plant_pos_problems(month_start, rows):
    """Adds or changes rows in a monthly POS file. Only some months get problems."""
    ym = month_start.strftime("%Y-%m")
    if ym == "2025-02":
        for row in rows:
            if row[2] == "AAR01":
                row[2] = " aar01"
                PLANTED["store_code_format"] += 1
    if ym == "2025-09":
        for i in range(3):
            rows.append([f"NYC02-250912-T{i + 1:03d}", "1", "NYC02", "12-09-2025 15:0" + str(i), "NK-HP-301", "1", "159.00", "0", "USD"])
    if ym == "2025-11":
        for i in range(8):
            rows.append([f"TRN01-251103-{i + 1:04d}", "1", "TRN01", "03-11-2025 09:1" + str(i), "NK-SPK-102", "1", "1999,00", "0", "DKK"])
    if ym == "2026-01":
        for i in range(6):
            rows.append([f"CPH01-260115-X{i + 1:03d}", "1", "CPH01", "15-01-2026 12:3" + str(i), "NK-HP-999", "1", "2499,00", "0", "DKK"])
    if ym == "2026-04":
        for i in range(5):
            rows.append([f"PAR01-260410-Z{i + 1:03d}", "1", "PAR01", "10-04-2026 11:2" + str(i), "NK-SPK-101", "0", "399,00", "0", "EUR"])
        for i in range(4):
            rows.append([f"MUC01-260411-P{i + 1:03d}", "1", "MUC01", "11-04-2026 14:4" + str(i), "NK-ACC-101", "1", "", "0", "EUR"])
    return rows

## Master data

In [ ]:
def products_csv():
    out = io.StringIO()
    w = csv.writer(out)
    w.writerow(["sku", "product_name", "category", "subcategory", "list_price_dkk", "unit_cost_dkk", "launch_date", "discontinued_date"])
    for p in PRODUCTS:
        sku, cat = p[0], p[2]
        if sku == "NK-ACC-104":  # planted: messy key and category
            sku, cat = "nk-acc-104 ", "accessories"
        w.writerow([sku, p[1], cat, p[3], p[4], round(p[4] * p[5], 2), p[6] or "", p[7] or ""])
    return out.getvalue()


def stores_csv():
    out = io.StringIO()
    w = csv.writer(out)
    w.writerow(["store_code", "store_name", "city", "country", "region", "channel", "currency", "opened_on", "closed_on"])
    for s in STORES:
        w.writerow(list(s[:8]) + [s[8] or ""])
    return out.getvalue()


def fx_csv(until):
    """Bank-day rates (DKK per 1 unit), a seeded random walk. Weekends and some holidays are missing."""
    out = io.StringIO()
    w = csv.writer(out)
    w.writerow(["rate_date", "currency", "dkk_per_unit"])
    r = random.Random(f"{seed}-fx")
    level = dict(FX_BASE)
    day = START - dt.timedelta(days=7)
    while day <= until:
        for cur in FX_BASE:
            drift = 0.0 if cur in ("EUR",) else r.gauss(0, 0.003)
            level[cur] = level[cur] * (1 + drift) if cur != "EUR" else 7.46 + r.uniform(-0.004, 0.004)
            if day.weekday() >= 5:
                continue
            if cur == "GBP" and dt.date(2025, 12, 24) <= day <= dt.date(2025, 12, 28):
                continue  # planted gap
            w.writerow([day.isoformat(), cur, f"{level[cur]:.4f}"])
        day += dt.timedelta(days=1)
    return out.getvalue()


def avg_line_value_dkk(day):
    items = [p for p in PRODUCTS if available(p, day)]
    total_w = sum(p[8] for p in items)
    return sum(p[8] * p[4] * (2 if p[2] == "Accessories" else 1) for p in items) / total_w


def targets_csv():
    """Monthly revenue targets per city, set once a year from expected demand plus planned growth."""
    out = io.StringIO()
    w = csv.writer(out)
    w.writerow(["month", "city", "target_dkk"])
    month = START
    while month <= dt.date(2027, 5, 1):
        nxt = (month.replace(day=28) + dt.timedelta(days=4)).replace(day=1)
        fy_start = dt.date(month.year if month.month >= 6 else month.year - 1, 6, 1)
        totals = {}
        day = month
        while day < nxt:
            alv = avg_line_value_dkk(day) * 0.91  # net of discounts, returns and cancellations
            # planned with last fiscal year's demand level plus TARGET_GROWTH
            plan_years = max((fy_start - START).days / 365.25, 0)
            for s in STORES:
                if not is_open(s, day):
                    continue
                if s[5] == "Online":
                    lam = 22 * 1.33 * season(day) * weekday_factor(day, True) / 2
                else:
                    lam = 4.5 * s[9] * 1.45 * season(day) * weekday_factor(day)
                base = (1 + CITY_GROWTH[s[2]]) ** plan_years if s[2] != "Online" else (1.12 ** plan_years)
                totals[s[2]] = totals.get(s[2], 0) + lam * alv * base * (1 + TARGET_GROWTH)
            day += dt.timedelta(days=1)
        for city in sorted(totals):
            w.writerow([month.isoformat(), city, round(totals[city], -3)])
        month = nxt
    return out.getvalue()

## Writers

In [ ]:
def write_text(path, text):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8", newline="") as f:
        f.write(text)


def pos_csv(rows):
    out = io.StringIO()
    w = csv.writer(out, delimiter=";")
    w.writerow(POS_HEADER)
    w.writerows(rows)
    return out.getvalue()


def json_lines(orders):
    return "".join(json.dumps(o, ensure_ascii=False) + "\n" for o in orders)


def write_master(root, until):
    write_text(f"{root}/master/products/products.csv", products_csv())
    write_text(f"{root}/master/stores/stores.csv", stores_csv())
    write_text(f"{root}/master/fx_rates/fx_rates.csv", fx_csv(until))
    write_text(f"{root}/master/sales_targets/sales_targets.csv", targets_csv())


def backfill(root, until):
    """History as monthly files, the way a first bulk export usually arrives."""
    files = 0
    month = START
    while month <= until:
        nxt = (month.replace(day=28) + dt.timedelta(days=4)).replace(day=1)
        pos, web = [], []
        day = month
        while day < nxt and day <= until:
            pos += pos_lines_for_day(day)
            web += web_export_for_day(day)
            day += dt.timedelta(days=1)
        pos = plant_pos_problems(month, pos)
        PLANTED["web_status_updates"] += sum(1 for o in web if o["exported_at"].endswith("T06:00:00Z"))
        write_text(f"{root}/pos/pos_{month:%Y-%m}.csv", pos_csv(pos))
        write_text(f"{root}/webshop/orders_{month:%Y-%m}.jsonl", json_lines(web))
        files += 2
        if month.strftime("%Y-%m") == "2025-03":
            write_text(f"{root}/pos/pos_2025-03_resend.csv", pos_csv(pos))
            files += 1
        month = nxt
    write_master(root, until)
    write_text(f"{root}/_generator_state.json", json.dumps({"last_day": until.isoformat()}))
    return files + 4


def next_day(root):
    """One more day, as daily files: what the pipeline sees every morning."""
    state = json.loads(open(f"{root}/_generator_state.json", encoding="utf-8").read())
    day = dt.date.fromisoformat(state["last_day"]) + dt.timedelta(days=1)
    write_text(f"{root}/pos/daily/pos_{day.isoformat()}.csv", pos_csv(pos_lines_for_day(day)))
    write_text(f"{root}/webshop/daily/orders_{day.isoformat()}.jsonl", json_lines(web_export_for_day(day)))
    write_text(f"{root}/master/fx_rates/fx_rates.csv", fx_csv(day))
    write_text(f"{root}/_generator_state.json", json.dumps({"last_day": day.isoformat()}))
    return day

In [ ]:
# RUN
if mode == "backfill":
    n = backfill(landing_root, dt.date.fromisoformat(end_date))
    print(f"wrote {n} files to {landing_root}, history {START} to {end_date}")
    print(json.dumps(PLANTED, indent=2))
elif mode == "next_day":
    day = next_day(landing_root)
    print(f"wrote the exports for {day}")
else:
    raise ValueError(f"unknown mode {mode}")